Q1.
(a) re.match(r"\d+", text) -> None — 문자열이 '강의는'으로 시작해서 매칭 실패
(b) .group() -> '2026-05-06'
(c) findall 그룹 없음 -> ['2026-05-06', '2026-05-18']
(d) findall 캡처 그룹 3개 -> [('2026','05','06'), ('2026','05','18')]
(e) findall 비캡처 그룹 -> ['2026-05-06', '2026-05-18']
추가: re.findall은 캡처 그룹 (...)이 있으면 튜플 리스트를, 없거나 (?:...) 비캡처이면 문자열 리스트를 반환한다.

Q2.
(a) 탐욕적 .+ -> '[T]!'
(b) 게으른 .+? -> '[T]안녕[T] [T]세상[T]!'
(c) [^>]+ -> '[T]안녕[T] [T]세상[T]!'
(d) 원시 문자열 r"<\1>" -> '수강생 <30>명, 조교 <3>명'
(e) 일반 문자열 "<\1>" -> '수강생 <\x01>명, 조교 <\x01>명' (역참조 실패)

(i) (a)의 .+는 탐욕적(greedy) 수량자라 <b>안녕</b> <i>세상</i> 전체를 하나로 매칭하지만, (b)의 .+?는 게으른(lazy) 수량자라 <b>, </b> 등 각 태그를 따로따로 매칭한다.
(ii) r"<\1>"는 원시 문자열이라 \1이 정규식 역참조로 해석되지만, "<\1>"는 일반 문자열이라 파이썬이 먼저 \1을 ASCII 제어 문자 \x01로 변환해버려 역참조가 동작하지 않는다.

In [2]:
import re
from collections import Counter

# 반복 사용 패턴 사전 컴파일
URL_PAT     = re.compile(r'https?://\S+')
HTML_PAT    = re.compile(r'<[^>]+>')
EMAIL_PAT   = re.compile(r'[\w.\-+]+@[\w.\-]+\.[a-zA-Z]{2,}')
PHONE_PAT   = re.compile(r'\d{2,4}-\d{3,4}-\d{4}')
MENTION_PAT = re.compile(r'@\w+')
HASHTAG_PAT = re.compile(r'#\w+')
JAMO_PAT    = re.compile(r'[\u3131-\u3163]+')
SPACE_PAT   = re.compile(r'\s+')


def clean_post(post: str) -> str:
    post = URL_PAT.sub(' ', post)       # 1. URL 제거
    post = HTML_PAT.sub('', post)       # 2. HTML 태그 제거
    post = EMAIL_PAT.sub('[이메일]', post) # 3. 이메일 마스킹
    post = PHONE_PAT.sub('[전화]', post)   # 3. 전화번호 마스킹
    post = MENTION_PAT.sub(' ', post)   # 4. 멘션 제거
    post = HASHTAG_PAT.sub(' ', post)   # 4. 해시태그 제거
    post = JAMO_PAT.sub('', post)       # 5. 자음/모음 제거
    post = SPACE_PAT.sub(' ', post).strip()  # 6. 공백 정리
    return post


def extract_hashtags(post: str) -> list[str]:
    return re.findall(r'#([\w\uAC00-\uD7A3]+)', post)


def analyze_posts(posts: list[str]) -> dict:
    cleaned = [clean_post(p) for p in posts]

    avg_length_after_clean: float = round(
        sum(len(c) for c in cleaned) / len(cleaned), 2
    )

    all_tags: list[str] = []
    for p in posts:
        all_tags.extend(extract_hashtags(p))
    hashtag_counts: dict[str, int] = dict(Counter(all_tags).most_common())

    masked_count: int = 0
    for p in posts:
        _, n_email = EMAIL_PAT.subn('[이메일]', p)
        _, n_phone = PHONE_PAT.subn('[전화]', p)
        masked_count += n_email + n_phone

    return {
        "posts_n":                len(posts),
        "avg_length_after_clean": avg_length_after_clean,
        "hashtag_counts":         hashtag_counts,
        "masked_count":           masked_count,
    }


# 확인
posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ "
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. "
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    " 여러 공백과\n\n\n줄바꿈이 많은 텍스트 ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

print("=== clean_post 결과 ===")
for i, p in enumerate(posts):
    print(f"[{i}] {clean_post(p)}")

print("\n=== analyze_posts 결과 ===")
print(analyze_posts(posts))

=== clean_post 결과 ===
[0] 오늘 수업 진짜 재밌었음!! 감사 자료:
[1] 팀플 어디서 모이지 카톡
[2] 중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!
[3] 여러 공백과 줄바꿈이 많은 텍스트
[4] 진짜 좋다

=== analyze_posts 결과 ===
{'posts_n': 5, 'avg_length_after_clean': 19.4, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}


=== clean_post 결과 ===
[0] 오늘 수업 진짜 재밌었음!! 감사 자료:
[1] 팀플 어디서 모이지 카톡
[2] 중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!
[3] 여러 공백과 줄바꿈이 많은 텍스트
[4] 진짜 좋다

=== analyze_posts 결과 ===
{'posts_n': 5, 'avg_length_after_clean': 19.4, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}

설명: 6단계 순서를 반드시 지켜야 하는 이유는, 단계 3(이메일 마스킹)을 단계 4(멘션 제거)보다 나중에 하면 @hong 같은 멘션이 먼저 공백으로 대체되어 이메일 주소의 @ 앞부분이 잘려나가므로 이메일 패턴이 제대로 매칭되지 않기 때문이다. 또한 단계 1(URL 제거)을 나중에 하면 URL 안의 문자열이 이메일·멘션 패턴과 오매칭될 수 있다.